In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir /content/embeddings
!cp /content/drive/MyDrive/tcga_UNI2h_embeddings/*.hdf5 /content/embeddings

# Load original embeddings matrix with metadata

In [ ]:
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

original_file = "embeddings/uni2h_embeddings_original.hdf5"

embeddings = []
metadata = []

with h5py.File(original_file, "r") as f:
    embedding_index = 0
    for category_name in f.keys():
        category_group = f[category_name]
        for patient_name in tqdm(category_group.keys()):
            patient_group = category_group[patient_name]
            for embedding_name in patient_group.keys():
                embedding_data = patient_group[embedding_name][()]  # load vector as np array

                # Store the vector
                embeddings.append(embedding_data)

                # Store metadata
                metadata.append({
                    "index": embedding_index,
                    # "category": category_name,
                    "patient": patient_name,
                    "embedding": embedding_name,
                    "rotation": 0
                })

                embedding_index += 1

# Convert to numpy + dataframe
embeddings = np.array(embeddings)
metadata_df = pd.DataFrame(metadata)

print("Embeddings shape:", embeddings.shape)
print(metadata_df.head())

100%|██████████| 666/666 [15:16<00:00,  1.38s/it]


Embeddings shape: (2036270, 1536)
   index                                            patient  \
0      0  TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...   
1      1  TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...   
2      2  TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...   
3      3  TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...   
4      4  TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...   

                                embedding  
0  TCGA-5M-AAT6-01Z-00-DX1_(100372,12166)  
1  TCGA-5M-AAT6-01Z-00-DX1_(100372,13180)  
2  TCGA-5M-AAT6-01Z-00-DX1_(100372,14194)  
3  TCGA-5M-AAT6-01Z-00-DX1_(100372,15207)  
4  TCGA-5M-AAT6-01Z-00-DX1_(100372,16221)  


In [ ]:
# metadata_df["rotation"] = 0
# metadata_df

,index,patient,embedding,rotation
0,0,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,12166)",0
1,1,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,13180)",0
2,2,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,14194)",0
3,3,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,15207)",0
4,4,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,16221)",0
...,...,...,...,...
2036265,2036265,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,71841)",0
2036266,2036266,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,72853)",0
2036267,2036267,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,73865)",0
2036268,2036268,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,74877)",0


# Create IVFPQ index with IP metric from original vectors

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 105.9 MB/s eta 0:00:00


In [ ]:
import faiss


# CREATE IVFPQ INDEX
# Suppose we already have N patch embeddings (dim = D)
N, D = embeddings.shape  # number of embeddings, embedding dimension

# Normalize embeddings (important for cosine similarity)
print("Normalizing...")
faiss.normalize_L2(embeddings)

# Create FAISS index (balanced settings for start)
nlist = 16_384      # number of clusters
m = 96              # PQ parameter (sub-vector size)
nbits = 8           # bits per subvector

quantizer = faiss.IndexFlatIP(D)
index = faiss.IndexIVFPQ(quantizer, D, nlist, m, nbits, faiss.METRIC_INNER_PRODUCT)

# Train the index (needed for IVF/PQ)
# Training
n_train = 500_000

print("Training...")
index.train(embeddings[:n_train])

print("Adding new embeddings...")
index.add(embeddings)

Normalizing...


# Add augmented embeddings to the index

In [ ]:
import gc

# Add embeddings from augmentations to index
augmented_files = [
    "embeddings/uni2h_embeddings_rot90.hdf5",
    "embeddings/uni2h_embeddings_rot180.hdf5",
    "embeddings/uni2h_embeddings_rot270.hdf5"
]

rotations = [90, 180, 270]

for augmented_file, rotation in zip(augmented_files, rotations):
    # Reset memory used by previous embeddings
    del embeddings
    del metadata
    gc.collect()

    # Create new embedding and metadata arrays
    embeddings = []
    metadata = []

    with h5py.File(augmented_file, "r") as f:
        for patient_name in tqdm(f.keys(), desc=f"{augmented_file} patients"):
            patient_group = f[patient_name]
            for embedding_name in patient_group.keys():
                embedding_data = patient_group[embedding_name][()]  # load vector as np array

                # Store the vector
                embeddings.append(embedding_data)

                # Store metadata
                metadata.append({
                    "index": embedding_index,
                    "patient": patient_name,
                    "embedding": embedding_name,
                    "rotation": rotation
                })

                embedding_index += 1

    # Convert to numpy + dataframe
    embeddings = np.array(embeddings)
    new_metadata_df = pd.DataFrame(metadata)

    # Join with metadata_df
    metadata_df = pd.concat([metadata_df, new_metadata_df], ignore_index=True)

    # Add new_embeddings to faiss index
    print("Adding new embeddings...")
    faiss.normalize_L2(embeddings)
    index.add(embeddings)

embeddings/uni2h_embeddings_rot90.hdf5 patients: 100%|██████████| 789/789 [18:51<00:00,  1.43s/it]


Adding new embeddings...


embeddings/uni2h_embeddings_rot180.hdf5 patients: 100%|██████████| 789/789 [19:20<00:00,  1.47s/it]


Adding new embeddings...


embeddings/uni2h_embeddings_rot270.hdf5 patients: 100%|██████████| 789/789 [18:56<00:00,  1.44s/it]


Adding new embeddings...


In [ ]:
metadata_df

,index,patient,embedding,rotation
0,0,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,12166)",0
1,1,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,13180)",0
2,2,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,14194)",0
3,3,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,15207)",0
4,4,TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-815...,"TCGA-5M-AAT6-01Z-00-DX1_(100372,16221)",0
...,...,...,...,...
8236947,8236947,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,71841)",270
8236948,8236948,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,72853)",270
8236949,8236949,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,73865)",270
8236950,8236950,TCGA-ZR-A9CJ-01Z-00-DX1.4A1AE826-FB78-4DD3-881...,"TCGA-ZR-A9CJ-01Z-00-DX1_(99162,74877)",270


# Save the index and metadata

In [ ]:
# Save index
index_file = "uni2h_IndexIVFPQ.faiss"
faiss.write_index(index, index_file)

print("Index saved to", index_file)

Index saved to uni2h_IndexIVFPQ.faiss


In [ ]:
!mv uni2h_IndexIVFPQ.faiss /content/drive/MyDrive/tcga_UNI2h_embeddings

In [ ]:
metadata_df.to_parquet("uni2h_IndexIVFPQ_patch_metadata.parquet", index=False)

In [ ]:
!mv uni2h_IndexIVFPQ_patch_metadata.parquet /content/drive/MyDrive/tcga_UNI2h_embeddings

# TEST FlatIP for 10 patients

In [ ]:
# # Get first 10 patient embeddings
# first_10_patients_df = metadata_df[metadata_df["patient"].isin(metadata_df["patient"].unique()[:10])].copy()
# print(first_10_patients_df.patient.unique())

# first_10_patients_indecies = first_10_patients_df["index"].values
# first_10_patients_embeddings = embeddings[first_10_patients_indecies].copy()
# first_10_patients_embeddings.shape

['TCGA-5M-AAT6-01Z-00-DX1.8834C952-14E3-4491-8156-52FC917BB014'
 'TCGA-A6-2686-01Z-00-DX1.0540a027-2a0c-46c7-9af0-7b8672631de7'
 'TCGA-A6-5661-01Z-00-DX1.bad2d858-11b4-4b9c-a720-daaae592cf48'
 'TCGA-A6-5665-01Z-00-DX1.3ad2c249-d138-4037-a59b-4747ce2b789a'
 'TCGA-A6-6653-01Z-00-DX1.e130666d-2681-4382-9e7a-4a4d27cb77a4'
 'TCGA-AA-3715-01Z-00-DX1.24d6e746-ad61-4587-a2b9-8903331b279c'
 'TCGA-AA-3811-01Z-00-DX1.369d7946-911e-4e97-8ae7-32ef12b6b106'
 'TCGA-AA-3821-01Z-00-DX1.019b3b1d-5c05-4be6-af25-6ee63475897e'
 'TCGA-AA-3845-01Z-00-DX1.20682536-a009-4184-a40b-cb889f37ad32'
 'TCGA-AA-3864-01Z-00-DX1.f6992bc7-ba05-4c30-9500-8f7b07b30f9a']


(12281, 1536)

In [ ]:
# import faiss

# # TEST FLAT IP INDEX
# N, D = first_10_patients_embeddings.shape  # number of embeddings, embedding dimension
# xq = first_10_patients_embeddings[0].reshape(1,-1).copy()  # Copy first vector before normalization

# # Normalize embeddings (important for cosine similarity)
# print("Normalizing...")
# faiss.normalize_L2(first_10_patients_embeddings)  # Normalize

# # Create index
# index_flat = faiss.IndexFlatIP(D)

# # 4) add data
# index_flat.add(first_10_patients_embeddings)

# # Save index
# flatip_index_file = "uni2h_IndexFlatIP.faiss"
# faiss.write_index(index_flat, flatip_index_file)
# print("Index saved to", flatip_index_file)

Normalizing...
Index saved to uni2h_IndexFlatIP.faiss


In [ ]:
# # 5) query: set
# faiss.normalize_L2(xq)
# k = 10
# D, I = index_flat.search(xq, k)  # approx distances and ids
# print("D shape:", D.shape, "I shape:", I.shape)
# print(I)
# print(D)

D shape: (1, 10) I shape: (1, 10)
[[   0 4335   33 4181 4129 4228   53 2483 3085 4070]]
[[1.         0.7075586  0.6776405  0.6742361  0.66285914 0.6410935
  0.6250518  0.62283015 0.6200999  0.6170816 ]]
